
# LSTM Forecasting Model

This notebook evaluates an LSTM model for one-month-ahead bilateral trade-flow forecasting for South Korea, Slovakia, and the USA. The model uses historical trade values and engineered temporal features to forecast March 2026.

Model selection and training use data available through February 2026. Forecasts are evaluated on the original trade-value scale using MAE, RMSE, and sMAPE, with results saved for the final model comparison.


## 1. Load Forecasting Data

Load the prepared training data and held-out March 2026 observations for each country.

In [2]:

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from src.dataset_loading import (
    load_forecasting_data,
    load_feature_data,
    SERIES_KEYS,
    TARGET,
)
from src.evaluation import evaluate_forecasts, save_model_results

training_data, test_data = load_forecasting_data()
feature_data = load_feature_data()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42
TEST_MONTH = pd.Timestamp("2026-03-01")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")


Device: cpu
PyTorch version: 2.14.1+cpu



## 2. Sequence and Feature Preparation

The LSTM uses the previous 12 months of observations to forecast the next month. The input sequences contain historical trade values and temporal features, while the target is the original `Value` on the forecast month.

Training and validation remain chronological. March 2026 is reserved for final evaluation, and scaling parameters will be estimated using training data only to prevent information leakage.


In [3]:

SEQUENCE_LENGTH = 12

LSTM_FEATURES = [
    TARGET,
    "month_of_year",
    "month_sin",
    "month_cos",
]

VALIDATION_MONTH = pd.Timestamp("2026-02-01")

print(f"Sequence length: {SEQUENCE_LENGTH} months")
print(f"Input features: {LSTM_FEATURES}")
print(f"Forecast month: {TEST_MONTH:%Y-%m}")
print(f"Validation month: {VALIDATION_MONTH:%Y-%m}")


Sequence length: 12 months
Input features: ['Value', 'month_of_year', 'month_sin', 'month_cos']
Forecast month: 2026-03
Validation month: 2026-02



## 3. Memory-Efficient Sequence Loading

Generate 12-month input sequences on demand using a PyTorch `Dataset`, rather than storing millions of overlapping sequences in memory. A `DataLoader` processes sequences in batches of 256, without shuffling, while preserving chronological order within each sequence and preventing sequences from crossing trade-series boundaries.



In [24]:

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import torch
 
SEQUENCE_LENGTH = 12
BATCH_SIZE = 256
EPOCHS = 15
PATIENCE = 5
VALIDATION_MONTH = pd.Timestamp("2026-02-01")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
LSTM_FEATURES = [TARGET, "month_of_year", "month_sin", "month_cos"]  # TARGET must stay at index 0
 
LSTM_CONFIGS = {
    "LSTM_1": {"hidden_size": 16, "num_layers": 1, "learning_rate": 1e-3},
    "LSTM_2": {"hidden_size": 32, "num_layers": 1, "learning_rate": 1e-3},
    "LSTM_3": {"hidden_size": 32, "num_layers": 2, "learning_rate": 5e-4},
}
 
 
class TradeSequenceDataset(Dataset):
    """Builds 12-month windows on demand. Assumes no gaps in the monthly series."""
 
    def __init__(self, df, feature_columns, target_column, target_month=None):
        self.series = []
        self.index = []
 
        df = df.sort_values(SERIES_KEYS + ["Month"])
 
        for _, group in df.groupby(SERIES_KEYS, sort=False, observed=True):
            group = group.sort_values("Month").reset_index(drop=True)
            features = group[feature_columns].to_numpy(dtype=np.float32)
            targets = group[target_column].to_numpy(dtype=np.float32)
            months = pd.to_datetime(group["Month"]).to_numpy()
 
            series_id = len(self.series)
            self.series.append((features, targets))
 
            for target_idx in range(SEQUENCE_LENGTH, len(group)):
                if target_month is None or pd.Timestamp(months[target_idx]) == target_month:
                    self.index.append((series_id, target_idx))
 
    def __len__(self):
        return len(self.index)
 
    def __getitem__(self, idx):
        series_id, target_idx = self.index[idx]
        features, targets = self.series[series_id]
        X = features[target_idx - SEQUENCE_LENGTH:target_idx]
        y = targets[target_idx]
        return torch.from_numpy(X.copy()), torch.tensor(y, dtype=torch.float32)
 
 
def prepare_log(df):
    """Sort and log-transform Value. Scaling happens afterwards, once."""
    df = df.sort_values(SERIES_KEYS + ["Month"]).copy()
    df[TARGET] = np.log1p(df[TARGET].clip(lower=0))
    return df
 
 
def inverse_value(scaled_values, scaler):
    """scaled log1p(Value) -> original Value. Value is column 0 of the scaler."""
    log_values = np.asarray(scaled_values) * scaler.scale_[0] + scaler.mean_[0]
    return np.maximum(np.expm1(log_values), 0.0)
 
 
lstm_scalers = {}
lstm_datasets = {}
lstm_loaders = {}
 
for country, df in training_data.items():
    df = prepare_log(df)
    train_df = df.loc[df["Month"] < VALIDATION_MONTH]
    val_history = df.loc[df["Month"] <= VALIDATION_MONTH].copy()
 
    # One scaler, fit on training rows only. TARGET (column 0) is scaled exactly once.
    scaler = StandardScaler().fit(train_df[LSTM_FEATURES])
    val_history[LSTM_FEATURES] = scaler.transform(val_history[LSTM_FEATURES])
 
    scaled_train = val_history.loc[val_history["Month"] < VALIDATION_MONTH]
 
    train_dataset = TradeSequenceDataset(scaled_train, LSTM_FEATURES, TARGET)
    val_dataset = TradeSequenceDataset(
        val_history, LSTM_FEATURES, TARGET, target_month=VALIDATION_MONTH
    )
 
    lstm_scalers[country] = scaler
    lstm_datasets[country] = {"train": train_dataset, "validation": val_dataset}
    lstm_loaders[country] = {
        # Shuffle windows across series; order WITHIN each window is untouched, so no leakage.
        "train": DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            generator=torch.Generator().manual_seed(42),
        ),
        "validation": DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False),
    }
 
    print(
        f"{country}: {len(train_dataset):,} training sequences; "
        f"{len(val_dataset):,} validation sequences"
    )
 

South Korea: 2,013,133 training sequences; 27,823 validation sequences
Slovakia: 2,540,561 training sequences; 21,973 validation sequences
USA: 3,414,628 training sequences; 35,843 validation sequences



## 4. LSTM Architecture and Training

Use a single-layer LSTM with 32 hidden units and a linear output layer to predict the next month's trade value. Train using huber loss and Adam optimization, with a fixed random seed for reproducibility. We will compare three predefined configurations rather than run an exhaustive hyperparameter search due to computational limitations. After each epoch, evaluate February 2026 validation MAE and retain the model weights with the lowest validation MAE. Early stopping limits unnecessary training, while March 2026 remains untouched until final evaluation.



In [25]:

import copy
from torch import nn

torch.manual_seed(42)
np.random.seed(42)

class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
        )
        self.output = nn.Linear(hidden_size, 1)
 
    def forward(self, x):
        output, _ = self.lstm(x)
        return self.output(output[:, -1, :]).squeeze(-1)
 
 
def validation_mae_original(model, val_loader, scaler):
    """MAE on the ORIGINAL trade-value scale."""
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            preds.append(model(X_batch.to(DEVICE)).cpu().numpy())
            actuals.append(y_batch.numpy())
    preds = inverse_value(np.concatenate(preds), scaler)
    actuals = inverse_value(np.concatenate(actuals), scaler)
    return float(np.mean(np.abs(preds - actuals)))
 
 
def build_model(config):
    return LSTMForecaster(
        input_size=len(LSTM_FEATURES),
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
    ).to(DEVICE)
 
 
def train_lstm_model(train_loader, val_loader, config, scaler, seed=42):
    torch.manual_seed(seed)          # same init for every config/country, independent of run order
    np.random.seed(seed)
 
    model = build_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    criterion = nn.HuberLoss()       # robust; trains on log scale
 
    best_mae, best_state, best_epoch, patience_counter = float("inf"), None, 0, 0
 
    for epoch in range(EPOCHS):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
 
        val_mae = validation_mae_original(model, val_loader, scaler)
        print(f"Epoch {epoch + 1}/{EPOCHS} | Validation MAE (original scale): {val_mae:,.2f}")
 
        if val_mae < best_mae:
            best_mae, best_state, best_epoch, patience_counter = (
                val_mae, copy.deepcopy(model.state_dict()), epoch + 1, 0
            )
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping.")
                break
 
    model.load_state_dict(best_state)
    return model, best_mae, best_epoch


## 5. Train and Validate LSTM Models

Train each predefined LSTM configuration using mini-batches without shuffling. 

In [26]:

lstm_models = {}
lstm_validation_results = []
 
for country in training_data:
    print(f"\n{'=' * 15} {country} {'=' * 15}")
    train_loader = lstm_loaders[country]["train"]
    val_loader = lstm_loaders[country]["validation"]
    scaler = lstm_scalers[country]

    lstm_models[country] = {}
    for config_name, config in LSTM_CONFIGS.items():
        print(f"\nTraining {config_name}")
        model, val_mae, best_epoch = train_lstm_model(train_loader, val_loader, config, scaler)
 
        lstm_models[country][config_name] = model
        lstm_validation_results.append({
            "Country": country,
            "Configuration": config_name,
            "Validation Month": "2026-02",
            "Validation MAE": val_mae,
            "Best Epoch": best_epoch,
            "N Training Sequences": len(lstm_datasets[country]["train"]),
            "N Validation Sequences": len(lstm_datasets[country]["validation"]),
        })
 
lstm_validation_results_df = pd.DataFrame(lstm_validation_results)
display(lstm_validation_results_df.sort_values(["Country", "Validation MAE"]).reset_index(drop=True))


=============== South Korea ===============

Training LSTM_1
Epoch 1/15 | Validation MAE (original scale): 1,708,261.19
Epoch 2/15 | Validation MAE (original scale): 1,739,667.69
Epoch 3/15 | Validation MAE (original scale): 1,474,218.54
Epoch 4/15 | Validation MAE (original scale): 1,539,653.44
Epoch 5/15 | Validation MAE (original scale): 1,386,410.77
Epoch 6/15 | Validation MAE (original scale): 1,491,130.27
Epoch 7/15 | Validation MAE (original scale): 1,379,044.64
Epoch 8/15 | Validation MAE (original scale): 1,304,003.54
Epoch 9/15 | Validation MAE (original scale): 1,303,469.41
Epoch 10/15 | Validation MAE (original scale): 1,389,080.69
Epoch 11/15 | Validation MAE (original scale): 1,331,379.18
Epoch 12/15 | Validation MAE (original scale): 1,370,388.86
Epoch 13/15 | Validation MAE (original scale): 1,417,071.27
Epoch 14/15 | Validation MAE (original scale): 1,354,959.82
Early stopping.

Training LSTM_2
Epoch 1/15 | Validation MAE (original scale): 1,327,415.46
Epoch 2/15 | Va

,Country,Configuration,Validation Month,Validation MAE,Best Epoch,N Training Sequences,N Validation Sequences
0,Slovakia,LSTM_3,2026-02,2.500276e+05,1,2540561,21973
1,Slovakia,LSTM_1,2026-02,2.836581e+05,1,2540561,21973
2,Slovakia,LSTM_2,2026-02,4.486934e+05,1,2540561,21973
3,South Korea,LSTM_2,2026-02,1.058063e+06,6,2013133,27823
4,South Korea,LSTM_3,2026-02,1.180645e+06,6,2013133,27823
5,South Korea,LSTM_1,2026-02,1.303469e+06,9,2013133,27823
6,USA,LSTM_1,2026-02,2.702092e+06,9,3414628,35843
7,USA,LSTM_3,2026-02,2.710258e+06,7,3414628,35843
8,USA,LSTM_2,2026-02,2.838419e+06,7,3414628,35843



## 6. Select the Best LSTM Configuration

For each country, select the LSTM configuration with the lowest February 2026 validation MAE. The selected configuration will be retrained using all eligible observations through February 2026 before forecasting March 2026.


In [27]:

best_lstm_configurations = (
    lstm_validation_results_df
    .sort_values(["Country", "Validation MAE"])
    .drop_duplicates(subset="Country")
    .set_index("Country")["Configuration"]
    .to_dict()
)

print("Selected LSTM configuration per country:")
for country, config_name in best_lstm_configurations.items():
    print(f"{country}: {config_name}")


Selected LSTM configuration per country:
Slovakia: LSTM_3
South Korea: LSTM_2
USA: LSTM_1



## 7.Final March 2026 Forecasts

For each country, retrain the selected LSTM configuration using eligible observations through February 2026. Use the previous 12 months of observed data to forecast March 2026, transform predictions back to the original trade-value scale, and evaluate against the held-out March targets.


In [28]:
final_lstm_predictions = {}
final_lstm_models = {}
lstm_results = []
 
for country, df in training_data.items():
    config_name = best_lstm_configurations[country]
    config = LSTM_CONFIGS[config_name]
    best_epoch = int(
        lstm_validation_results_df.loc[
            (lstm_validation_results_df["Country"] == country)
            & (lstm_validation_results_df["Configuration"] == config_name),
            "Best Epoch",
        ].iloc[0]
    )
 

    df = prepare_log(df)
    train_df = df.loc[df["Month"] <= VALIDATION_MONTH]
 
    scaler = StandardScaler().fit(train_df[LSTM_FEATURES])
    scaled_train = train_df.copy()
    scaled_train[LSTM_FEATURES] = scaler.transform(train_df[LSTM_FEATURES])
 
    train_dataset = TradeSequenceDataset(scaled_train, LSTM_FEATURES, TARGET)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(42),
    )
 
 
    torch.manual_seed(42)
    np.random.seed(42)
    model = build_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    criterion = nn.HuberLoss()
 
    model.train()
    for epoch in range(best_epoch):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
 

    test_df = test_data[country].copy()
    test_keys = set(map(tuple, test_df[SERIES_KEYS].to_numpy()))
 
    sequences, sequence_keys = [], []
    for key, group in scaled_train.groupby(SERIES_KEYS, sort=False, observed=True):
        key = key if isinstance(key, tuple) else (key,)
        if key not in test_keys or len(group) < SEQUENCE_LENGTH:
            continue
        group = group.sort_values("Month")
        # With no gaps, the last row must be Feb 2026 so the window is Mar 2025 - Feb 2026.
        if pd.Timestamp(group["Month"].iloc[-1]) != VALIDATION_MONTH:
            continue
        sequences.append(group[LSTM_FEATURES].to_numpy(dtype=np.float32)[-SEQUENCE_LENGTH:])
        sequence_keys.append(key)
 
    if not sequences:
        raise ValueError(f"No March sequences available for {country}.")
 
    X_march = torch.tensor(np.asarray(sequences), dtype=torch.float32).to(DEVICE)
 
    model.eval()
    scaled_predictions = []
    with torch.no_grad():
        for start in range(0, len(X_march), BATCH_SIZE):
            scaled_predictions.append(model(X_march[start:start + BATCH_SIZE]).cpu().numpy())
    predictions = inverse_value(np.concatenate(scaled_predictions), scaler)  # original scale, >= 0
 
    keys_df = pd.DataFrame(sequence_keys, columns=SERIES_KEYS)
    final_df = keys_df.merge(
        test_df[SERIES_KEYS + [TARGET]], on=SERIES_KEYS, how="inner", validate="one_to_one"
    )
    # merge preserves keys_df order for a one-to-one inner merge, but assert it anyway
    assert len(final_df) == len(keys_df) == len(predictions)
    final_df["lstm_prediction"] = predictions
 
    final_lstm_predictions[country] = final_df
    final_lstm_models[country] = model
 
    lstm_results.append(
        evaluate_forecasts(
            country=country,
            model_name=f"LSTM ({config_name})",
            actual=final_df[TARGET],
            predicted=final_df["lstm_prediction"],
            forecast_month="2026-03",
        )
    )
 
lstm_results_df = save_model_results(lstm_results, model_name="LSTM")
display(lstm_results_df)
 

Saved model results to: C:\Transfer\InterdisciplinaryProject\results\lstm_results.csv


,Country,Model,Forecast Month,N Series,MAE,RMSE,sMAPE (%)
0,Slovakia,LSTM (LSTM_3),2026-03,21973,2.400967e+05,3.217984e+06,79.852074
1,South Korea,LSTM (LSTM_2),2026-03,27823,1.616569e+06,4.507207e+07,108.689753
2,USA,LSTM (LSTM_1),2026-03,35843,3.764889e+06,7.274987e+07,84.354281



## Summary

This notebook evaluated LSTM models for one-month-ahead bilateral trade-flow forecasting for South Korea, Slovakia, and the USA. Three predefined configurations were compared using February 2026 validation MAE, and the selected configurations were refitted using data available through February 2026.

March 2026 forecasts were evaluated on the original trade-value scale using MAE, RMSE, and sMAPE. The results were saved to the `results/` directory for comparison with the baseline, statistical, and XGBoost models.

The experiment used fixed-length 12-month sequences and mini-batch training without shuffling. Model performance should be interpreted cautiously because evaluation covers only one held-out month, and computational constraints limited the configuration search.
